# TrajGPT: time-conditioned pretraining and inference

This notebook uses the current project code for PopHR:

1. Inspect the data and teacher-forcing alignment.
2. Optionally pretrain or resume with `run_pretrain.py`.
3. Load a saved model, configuration and vocabulary.
4. Use **50 observed events and their times** to generate the remaining PheCodes at **given future event times**.
5. Evaluate the generated trajectory.

Only PheCodes are predicted. Future times are supplied conditions. Irregular intervals and repeated times are allowed. The SRA backbone and frozen 200-dimensional PheCode embeddings are retained; this workflow does not use ODE inference.

## 1. Setup

Start the notebook kernel in the `TrajGPT` project directory. Run the cells in order.
The default configuration loads the existing epoch-3 temporal checkpoint. Set `RUN_PRETRAIN = True` to create a new training run. Device selection automatically tries CUDA, MPS, then CPU.

Data and pretrained weights are not distributed with this repository. Provide the PopHR CSV and PheCode embedding CSV in `data/` locally. On a fresh checkout, set `RUN_PRETRAIN = True` to create a checkpoint, or set `CHECKPOINT` to your own compatible temporal checkpoint.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

from data.data_pipeline import DATA_PATH, DATA_ROW_LIMIT, data_provider
from run_pretrain import main as pretrain_main
from run_inference import (
    evaluate_predictions, phenotype_names, select_test_patient, trajectory_rows,
)
from training_utils import SOURCE_ROOT, load_checkpoint, seed_everything, select_device

DEVICE = "auto"
BATCH_SIZE = 8
PREFIX_LENGTH = 50
PATIENT_INDEX = None  # None selects the longest eligible patient in the saved test split.
TOP_K = 5
CHECKPOINT = SOURCE_ROOT / "checkpoints/time_pretraining_resume_validation_20261122/best.pt"

RUN_PRETRAIN = False
RESUME_FROM = None  # To resume, set this to a saved last.pt and increase TRAIN_EPOCHS.
TRAIN_EPOCHS = 2
LEARNING_RATE = 3e-5

seed_everything(2021)
torch.set_num_threads(4)
device = select_device(DEVICE)
print(f"Project: {SOURCE_ROOT}\nDevice: {device}")

## 2. Inspect the existing PopHR slice

The loader still uses the **first 10,000 CSV rows**, not 10,000 patients. Each record has a patient `id`, an event `date`, a `PheCode`, and `age_at_diag` in months. Patient records are sorted by date, keeping a stable order for events on the same date. The row limit can cut off a patient's history; it is unchanged in this notebook.

In [ ]:
data_preview = pd.read_csv(DATA_PATH, nrows=DATA_ROW_LIMIT)
print(f"Rows: {len(data_preview):,}; patients: {data_preview['id'].nunique()}")
display(data_preview[["date", "PheCode", "age_at_diag"]].head(5))

## 3. Optional pretraining or resume

This cell calls the existing training entry point. It does not duplicate the optimizer, loss, checkpoint or validation logic.

- Fresh runs use historical-time RoPE and target-time conditioning (`time_conditioned`, months, scale 1.0).
- The default model has four SRA layers, four heads and fixed 200-dimensional embeddings.
- `TRAIN_EPOCHS` is the **total** target epoch count. For example, resume epoch 3 with `TRAIN_EPOCHS = 4`.
- On resume, the checkpoint's temporal configuration and patient splits are reused.
- Each run gets a new directory with `last.pt`, configuration, splits and logs; `best.pt` is saved when validation improves.

After optional training, this notebook loads that run's `last.pt`. Leave `RUN_PRETRAIN = False` to use `CHECKPOINT` without training.

In [ ]:
if RUN_PRETRAIN:
    training_args = [
        "--device", DEVICE,
        "--batch_size", str(BATCH_SIZE),
        "--train_epochs", str(TRAIN_EPOCHS),
    ]
    if RESUME_FROM is None:
        training_args += [
            "--time_mode", "time_conditioned", "--time_scale", "1.0",
            "--learning_rate", str(LEARNING_RATE),
        ]
    else:
        training_args += ["--resume", str(RESUME_FROM)]
    run_dir = pretrain_main(training_args)
    CHECKPOINT = run_dir / "last.pt"
else:
    print("Pretraining skipped; loading the configured checkpoint.")

## 4. Load the pretrained model

`load_checkpoint()` restores the model and its fixed embedding table, vocabulary and temporal configuration. Inference uses the **pretraining output head**; no new forecast head is created.

In [ ]:
model, checkpoint = load_checkpoint(CHECKPOINT, device=device)
model.eval()
config = checkpoint["config"]
print(f"Checkpoint: {CHECKPOINT}\nSaved epoch: {checkpoint['epoch']}")
print(f"Embedding trainable: {model.token_embedding.emb.weight.requires_grad}")
display(pd.Series({key: config[key] for key in [
    "time_mode", "time_unit", "time_scale", "seq_len", "num_layers", "num_heads", "d_model",
]}))

## 5. Data loading and teacher-forcing alignment

Load the **saved patient splits**. Each batch is now a dictionary:

| Key | Meaning |
|---|---|
| `input_ids` | SOS followed by preceding real events |
| `input_time` | Each input event's own time; SOS uses the first retained event time |
| `labels` | The next-event PheCodes |
| `target_time` | Each label's target time |
| `attention_mask` | Valid positions; right padding is false |

For labels `C, D, E, F`, the inputs are `SOS, C, D, E`. Their times move together. Labels and target times are not shifted. The model performs **no additional shift**.

A training window retains the latest 256 target events and matching times. PAD is code 0 and is excluded from loss. SOS is -1.

In [ ]:
(
    train_dataset, train_loader, valid_dataset, valid_loader, test_dataset, test_loader,
) = data_provider(
    batch_size=BATCH_SIZE,
    max_length=config["seq_len"],
    split_ids=checkpoint["split_ids"],
    time_mode=config["time_mode"],
)
print(f"Patients: train={len(train_dataset)}, valid={len(valid_dataset)}, test={len(test_dataset)}")
batch = next(iter(train_loader))
display(pd.DataFrame([
    {"field": key, "shape": tuple(value.shape), "dtype": str(value.dtype)}
    for key, value in batch.items()
]))
valid_positions = batch["attention_mask"][0]
alignment = pd.DataFrame({
    "input_event": batch["input_ids"][0, valid_positions].tolist(),
    "input_time": batch["input_time"][0, valid_positions].tolist(),
    "target_event": batch["labels"][0, valid_positions].tolist(),
    "target_time": batch["target_time"][0, valid_positions].tolist(),
})
display(alignment.head(8))

## 6. Current model forward interface

The model returns a dictionary with `logits` and, when labels are supplied, `loss`. Pass both history and target times. The loss is PheCode cross-entropy with PAD ignored. This demonstration scores one batch without updating the loaded model.

In [ ]:
device_batch = {key: value.to(device) for key, value in batch.items()}
with torch.inference_mode():
    output = model(
        X=device_batch["input_ids"],
        input_time=device_batch["input_time"],
        target_time=device_batch["target_time"],
        y=device_batch["labels"],
        retention_mask=device_batch["attention_mask"],
    )
print(f"Logits shape: {tuple(output['logits'].shape)}")
print(f"Valid-token cross-entropy: {output['loss'].item():.6f}")

## 7. Training log

Read the loss history saved beside this checkpoint. A resumed run's history contains that run's resumed epochs. These are teacher-forced training/validation losses; they are not autoregressive trajectory accuracy.

In [ ]:
history = pd.DataFrame(json.loads(CHECKPOINT.with_name("history.json").read_text()))
display(history[["epoch", "train_loss", "valid_loss"]])
ax = history.plot(x="epoch", y=["train_loss", "valid_loss"], marker="o", figsize=(7, 3))
ax.set_ylabel("Cross-entropy per valid token")
ax.set_title("Saved training run")
plt.tight_layout()
plt.show()

## 8. Select a held-out sequence

Use the raw chronological event list, not the shifted/truncated training sample. By default, selection uses the longest test sequence with more than 50 events. Set `PATIENT_INDEX` to choose a particular test-table index. Selection does not use prediction accuracy.

The first 50 real events exclude SOS and PAD. The remaining **times** are query conditions; the remaining **PheCodes** are held out for evaluation.

In [ ]:
events, times, selection = select_test_patient(
    checkpoint, model.context_length, PREFIX_LENGTH, PATIENT_INDEX,
)
history_codes = events[:PREFIX_LENGTH]
history_times = times[:PREFIX_LENGTH]
future_times = times[PREFIX_LENGTH:]
display(pd.Series({
    "test_patient_index": selection["test_patient_index"],
    "total_events": len(events),
    "observed_events": len(history_codes),
    "future_queries": len(future_times),
    "eligible_test_patients": selection["eligible_test_patients"],
}))

## 9. Autoregressive inference at the given times

Call `TrajGPT.inference()` directly. For the first step it uses `SOS, x1, ..., x50` and queries `t51`. It then appends `(predicted x51, given t51)` before querying `t52`.

Only generated PheCodes are fed back after the observed prefix. Each step recomputes the prefix through the trained output head, without a recurrent cache. If necessary, the window keeps SOS plus the latest 255 events, truncating events and times together.

`top_k=5` reports five candidates. Decoding still feeds back only the highest-scoring valid PheCode; PAD cannot be generated.

In [ ]:
predictions = model.inference(
    history_codes=history_codes,
    history_times=history_times,
    future_times=future_times,
    top_k=TOP_K,
)
print(f"Generated {len(predictions['phecodes'])} PheCodes at the supplied future times.")

## 10. Evaluate the generated trajectory

The true future diagnoses are used **after generation**, only for comparison. Top-k accuracy is measured along the same greedy trajectory, not along k separate trajectories. Candidate probabilities are normalized over valid PheCodes with PAD excluded.

In [ ]:
true_future_codes = events[PREFIX_LENGTH:]
metrics = evaluate_predictions(predictions, true_future_codes)
comparison = pd.DataFrame(trajectory_rows(
    predictions, history_times, PREFIX_LENGTH, phenotype_names(),
    true_codes=true_future_codes,
))
display(pd.Series({
    "evaluated_events": metrics["n_events"],
    "top1_correct": metrics["correct"],
    "top1_accuracy": metrics["top1_accuracy"],
    "top_k": metrics["top_k"],
    "top_k_correct": metrics["top_k_correct"],
    "top_k_accuracy": metrics["top_k_accuracy"],
}))
display(comparison[[
    "event_index", "query_time", "delta_months",
    "true_phecode", "true_phenotype", "predicted_phecode", "predicted_phenotype",
    "correct", "top_k_correct",
]].head(15))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(comparison["step"], comparison["delta_months"], marker=".")
axes[0].set(xlabel="Future event step", ylabel="Months since preceding event",
            title="Given irregular event intervals")
steps = comparison["step"]
axes[1].plot(steps, comparison["correct"].cumsum() / steps, label="Top-1")
axes[1].plot(steps, comparison["top_k_correct"].cumsum() / steps, label=f"Top-{metrics['top_k']}")
axes[1].set(xlabel="Generated events", ylabel="Cumulative accuracy", ylim=(-0.02, 1.02),
            title="Accuracy along the generated trajectory")
axes[1].legend()
plt.tight_layout()
plt.show()

## Save a report with the external entry point

`run_inference.py` uses the same model method and evaluation functions. To save JSON and per-event CSV files, run this in a new cell. Choose an unused output name; existing results are not overwritten.

```python
from run_inference import main as inference_main

report = inference_main([
    "--checkpoint", str(CHECKPOINT),
    "--device", DEVICE,
    "--patient_index", str(selection["test_patient_index"]),
    "--prefix_length", str(PREFIX_LENGTH),
    "--top_k", str(TOP_K),
    "--output", str(SOURCE_ROOT / "results/notebook_inference.json"),
])
```

This single-patient example checks the workflow. A short-trained checkpoint and one patient do not establish general prediction quality. Query times and the number of future queries are supplied, not predicted.

## Optional: visualize the fixed embedding table

The embedding table is **frozen during current pretraining**. UMAP therefore visualizes the supplied PheCode vectors stored in the checkpoint, not new token embeddings learned by this training run. Cluster appearance alone is not evidence of predictive performance.

Set `RUN_UMAP = True` to run this optional visualization. It requires `umap-learn` and is skipped by default.

In [ ]:
RUN_UMAP = False
if RUN_UMAP:
    from umap import UMAP

    codes = model.token_embedding.phecodes
    vectors = model.token_embedding.emb.weight[:len(codes)].detach().cpu().numpy()
    coordinates = UMAP(
        n_neighbors=5, n_components=2, metric="cosine", min_dist=0.2, random_state=2021,
    ).fit_transform(vectors)
    definitions = pd.read_csv(SOURCE_ROOT / "data/phecode_definitions1.2.csv")
    points = pd.DataFrame({"phecode": codes, "UMAP 1": coordinates[:, 0], "UMAP 2": coordinates[:, 1]})
    points = points.merge(definitions[["phecode", "category"]], on="phecode", how="left")
    points["category"] = points["category"].fillna("Unknown")
    fig, ax = plt.subplots(figsize=(7, 5))
    for category, group in points.groupby("category"):
        ax.scatter(group["UMAP 1"], group["UMAP 2"], label=category, s=15, alpha=0.7)
    ax.set(title="Fixed PheCode embeddings", xlabel="UMAP 1", ylabel="UMAP 2")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize="small")
    plt.tight_layout()
    plt.show()